# LeetCode 836 — Rectangle Overlap bằng Python

**Link đề:** https://leetcode.com/problems/rectangle-overlap/
**Độ khó:** Easy · **Chủ đề:** Math, Geometry

```python
class Solution:
    def isRectangleOverlap(self, rec1: list[int], rec2: list[int]) -> bool:
        ...
```

## Mục tiêu học
1. Đọc tọa độ hình chữ nhật song song trục `[x1, y1, x2, y2]` (góc dưới-trái, góc trên-phải).
2. Hiểu "chồng lấn" nghĩa là **diện tích giao > 0** — chạm cạnh/chạm góc **không** tính.
3. Học kỹ thuật then chốt: **chiếu xuống từng trục** — hai hình chữ nhật chồng nhau ⇔ hai **đoạn** trên trục x chồng nhau **và** hai đoạn trên trục y chồng nhau.
4. Bốn cách: đếm ô (brute force minh họa) → điều kiện phủ định "nằm hẳn một phía" → giao đoạn `min/max` → tính diện tích giao.
5. Phân biệt `<` và `<=` ở biên, test các ca chạm cạnh/góc/chứa nhau, và mở rộng sang *Rectangle Area (223)*.

## 1. Đề bài và ví dụ

Hình chữ nhật song song trục được biểu diễn bởi `[x1, y1, x2, y2]`: `(x1, y1)` là góc **dưới-trái**, `(x2, y2)` là góc **trên-phải** (nên `x1 < x2`, `y1 < y2`).

Hai hình chữ nhật **chồng lấn (overlap)** nếu phần giao của chúng có **diện tích dương**. Chỉ chạm nhau ở cạnh hoặc góc thì **không** tính.

| rec1 | rec2 | Kết quả | Vì sao |
|---|---|---|---|
| `[0,0,2,2]` | `[1,1,3,3]` | `True` | giao là hình vuông `[1,1,2,2]`, diện tích 1 |
| `[0,0,1,1]` | `[1,0,2,1]` | `False` | chỉ chung cạnh `x = 1`, diện tích 0 |
| `[0,0,1,1]` | `[2,2,3,3]` | `False` | tách rời hoàn toàn |

**Constraints:** mỗi list dài 4; tọa độ trong `[-10^9, 10^9]`; mỗi hình có diện tích khác 0.

> Tọa độ lớn tới `10^9` ⇒ **không thể** "vẽ" từng ô để kiểm tra; cần công thức `O(1)`.

In [1]:
rec1 = [0, 0, 2, 2]
x1, y1, x2, y2 = rec1          # unpacking: tách list 4 phần tử thành 4 biến
print("góc dưới-trái:", (x1, y1), "| góc trên-phải:", (x2, y2))
print("rộng:", x2 - x1, "| cao:", y2 - y1, "| diện tích:", (x2 - x1) * (y2 - y1))

góc dưới-trái: (0, 0) | góc trên-phải: (2, 2)
rộng: 2 | cao: 2 | diện tích: 4


## 2. Nền tảng Python và toán cần dùng

### 2.1. Unpacking — tách list thành nhiều biến

`a, b, c, d = [1, 2, 3, 4]` gán lần lượt. Số biến phải bằng số phần tử, nếu không sẽ `ValueError`.

In [2]:
a, b, c, d = [1, 2, 3, 4]
print(a, b, c, d)
try:
    p, q = [1, 2, 3]
except ValueError as e:
    print("ValueError:", e)

1 2 3 4
ValueError: too many values to unpack (expected 2)


### 2.2. `max`, `min` với nhiều đối số

`max(3, 7)` = 7, `min(3, 7)` = 3. Ta sẽ dùng: đầu trái của phần giao = `max` hai đầu trái; đầu phải = `min` hai đầu phải.

In [3]:
print(max(3, 7), min(3, 7), max(-5, -2, 0), min(10**9, -10**9))

7 3 0 -1000000000


### 2.3. Đoạn trên trục số và giao của hai đoạn

Đoạn `[a1, a2]` và `[b1, b2]` (với `a1 < a2`, `b1 < b2`). Phần giao (nếu có) là `[max(a1, b1), min(a2, b2)]`. Độ dài giao = `min(a2, b2) - max(a1, b1)`:
- `> 0`: chồng nhau thật sự,
- `= 0`: chỉ chạm nhau tại một điểm,
- `< 0`: tách rời.

In [4]:
def overlap_len(a1, a2, b1, b2):
    return min(a2, b2) - max(a1, b1)

for seg in [(0, 2, 1, 3), (0, 1, 1, 2), (0, 1, 2, 3), (0, 5, 1, 2)]:
    print(seg, "-> độ dài giao =", overlap_len(*seg))

(0, 2, 1, 3) -> độ dài giao = 1
(0, 1, 1, 2) -> độ dài giao = 0
(0, 1, 2, 3) -> độ dài giao = -1
(0, 5, 1, 2) -> độ dài giao = 1


`overlap_len(*seg)`: dấu `*` **trải** (unpack) tuple `seg` thành các đối số riêng — `f(*(0, 2, 1, 3))` giống `f(0, 2, 1, 3)`.

### 2.4. Toán tử logic `and`, `or`, `not`

`A and B` đúng khi cả hai đúng; `A or B` đúng khi ít nhất một đúng; `not A` đảo. **Định luật De Morgan:** `not (A or B)` = `(not A) and (not B)`. Ta sẽ dùng định luật này ở cách 2.

In [5]:
for A in [False, True]:
    for B_ in [False, True]:
        print(A, B_, "| not(A or B) =", not (A or B_), "| (not A) and (not B) =", (not A) and (not B_))

False False | not(A or B) = True | (not A) and (not B) = True
False True | not(A or B) = False | (not A) and (not B) = False
True False | not(A or B) = False | (not A) and (not B) = False
True True | not(A or B) = False | (not A) and (not B) = False


## 3. Cách 1 — Brute force: đếm ô vuông đơn vị chung (chỉ để minh họa)

Nếu tọa độ là số nguyên **nhỏ**, ta có thể chia mặt phẳng thành các ô `1 × 1`. Ô có góc dưới-trái `(x, y)` nằm trong hình `[x1, y1, x2, y2]` khi `x1 <= x < x2` và `y1 <= y < y2`. Hai hình chồng nhau ⇔ có ít nhất một ô thuộc cả hai.

Chỉ dùng được khi tọa độ nhỏ — với `10^9` thì số ô khổng lồ. Nhưng nó là **oracle** (lời giải chắc chắn đúng) rất tốt để kiểm tra các công thức sau.

In [6]:
def overlap_grid(rec1: list[int], rec2: list[int]) -> bool:
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    lo_x, hi_x = min(ax1, bx1), max(ax2, bx2)
    lo_y, hi_y = min(ay1, by1), max(ay2, by2)
    for x in range(lo_x, hi_x):
        for y in range(lo_y, hi_y):
            in_a = ax1 <= x < ax2 and ay1 <= y < ay2
            in_b = bx1 <= x < bx2 and by1 <= y < by2
            if in_a and in_b:
                return True
    return False

print(overlap_grid([0, 0, 2, 2], [1, 1, 3, 3]))
print(overlap_grid([0, 0, 1, 1], [1, 0, 2, 1]))
print(overlap_grid([0, 0, 1, 1], [2, 2, 3, 3]))

True
False
False


`ax1 <= x < ax2` là **so sánh nối chuỗi** trong Python, nghĩa là `ax1 <= x and x < ax2`.

### Nút thắt
Số ô cần duyệt ≈ (độ rộng vùng bao) × (độ cao vùng bao). Với tọa độ `±10^9` là `~4·10^18` ô — không thể chạy. Ta cần suy nghĩ hình học.

## 4. Insight — Khi nào hai hình chữ nhật **không** chồng nhau?

Dễ hơn là nghĩ về trường hợp **không** chồng. Hai hình (song song trục) không chồng ⇔ một hình nằm **hẳn** về một phía của hình kia:

1. rec1 nằm **bên trái** rec2: `rec1.x2 <= rec2.x1`
2. rec1 nằm **bên phải** rec2: `rec1.x1 >= rec2.x2`
3. rec1 nằm **bên dưới** rec2: `rec1.y2 <= rec2.y1`
4. rec1 nằm **bên trên** rec2: `rec1.y1 >= rec2.y2`

Dùng `<=` / `>=` (có dấu bằng) vì chạm cạnh cũng là "không chồng".

Vậy: **chồng nhau ⇔ không xảy ra điều nào trong 4 điều trên.**

```text
     y
     ^        +-----+
     |        | rec2|
     |  +---+ +-----+      rec1.x2 <= rec2.x1  -> rec1 hẳn bên trái -> KHÔNG chồng
     |  |r1 |
     |  +---+
     +--------------------> x
```

In [7]:
rec1, rec2 = [0, 0, 1, 1], [1, 0, 2, 1]
ax1, ay1, ax2, ay2 = rec1
bx1, by1, bx2, by2 = rec2
print("trái :", ax2 <= bx1)
print("phải :", ax1 >= bx2)
print("dưới :", ay2 <= by1)
print("trên :", ay1 >= by2)

trái : True
phải : False
dưới : False
trên : False


## 5. Cách 2 — Phủ định "nằm hẳn một phía" (lời giải chính)

**Đoạn 1:** tách tọa độ bằng unpacking.

**Đoạn 2:** tính 4 điều kiện tách rời.

**Đoạn 3:** `return not (left or right or below or above)`.

In [8]:
def overlap_separation(rec1: list[int], rec2: list[int]) -> bool:
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    left  = ax2 <= bx1      # rec1 hẳn bên trái rec2
    right = ax1 >= bx2      # rec1 hẳn bên phải rec2
    below = ay2 <= by1      # rec1 hẳn bên dưới rec2
    above = ay1 >= by2      # rec1 hẳn bên trên rec2
    return not (left or right or below or above)

print(overlap_separation([0, 0, 2, 2], [1, 1, 3, 3]))
print(overlap_separation([0, 0, 1, 1], [1, 0, 2, 1]))
print(overlap_separation([0, 0, 1, 1], [2, 2, 3, 3]))

True
False
False


## 6. Cách 3 — Chiếu xuống hai trục và giao đoạn

Áp dụng De Morgan cho cách 2:

```text
not (left or right)  =  (ax2 > bx1) and (ax1 < bx2)   ⇔  min(ax2, bx2) > max(ax1, bx1)
not (below or above) =  (ay2 > by1) and (ay1 < by2)   ⇔  min(ay2, by2) > max(ay1, by1)
```

Nghĩa hình học: **chiếu** hai hình chữ nhật xuống trục x được hai đoạn `[ax1, ax2]`, `[bx1, bx2]`; chiếu xuống trục y được hai đoạn khác. Hai hình chồng nhau ⇔ **cả hai** cặp đoạn đều chồng nhau với độ dài giao **dương**.

Kỹ thuật "tách bài toán 2 chiều thành hai bài toán 1 chiều độc lập" rất mạnh, hay gặp ở hình học song song trục.

In [9]:
def overlap_projection(rec1: list[int], rec2: list[int]) -> bool:
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    x_overlap = min(ax2, bx2) - max(ax1, bx1)
    y_overlap = min(ay2, by2) - max(ay1, by1)
    return x_overlap > 0 and y_overlap > 0

print(overlap_projection([0, 0, 2, 2], [1, 1, 3, 3]))
print(overlap_projection([0, 0, 1, 1], [1, 0, 2, 1]))
print(overlap_projection([0, 0, 1, 1], [2, 2, 3, 3]))

True
False
False


## 7. TRACE từng bước

### 7.1. Bảng trace cho cả 3 ví dụ đề và một ca "chứa nhau"

| rec1 | rec2 | max(x1) | min(x2) | giao x | max(y1) | min(y2) | giao y | Kết quả |
|---|---|---|---|---|---|---|---|---|
| [0,0,2,2] | [1,1,3,3] | 1 | 2 | 1 | 1 | 2 | 1 | True |
| [0,0,1,1] | [1,0,2,1] | 1 | 1 | **0** | 0 | 1 | 1 | False (chạm cạnh) |
| [0,0,1,1] | [2,2,3,3] | 2 | 1 | −1 | 2 | 1 | −1 | False |
| [0,0,10,10] | [2,3,4,5] | 2 | 4 | 2 | 3 | 5 | 2 | True (chứa nhau) |

In [10]:
def trace_overlap(rec1, rec2):
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    lx, rx = max(ax1, bx1), min(ax2, bx2)
    ly, ry = max(ay1, by1), min(ay2, by2)
    print(f"rec1={rec1} rec2={rec2}")
    print(f"  trục x: đoạn [{ax1},{ax2}] và [{bx1},{bx2}] -> giao [{lx},{rx}] dài {rx - lx}")
    print(f"  trục y: đoạn [{ay1},{ay2}] và [{by1},{by2}] -> giao [{ly},{ry}] dài {ry - ly}")
    res = rx - lx > 0 and ry - ly > 0
    print("  =>", res)
    return res

for r1, r2 in [([0, 0, 2, 2], [1, 1, 3, 3]), ([0, 0, 1, 1], [1, 0, 2, 1]),
               ([0, 0, 1, 1], [2, 2, 3, 3]), ([0, 0, 10, 10], [2, 3, 4, 5])]:
    trace_overlap(r1, r2)

rec1=[0, 0, 2, 2] rec2=[1, 1, 3, 3]
  trục x: đoạn [0,2] và [1,3] -> giao [1,2] dài 1
  trục y: đoạn [0,2] và [1,3] -> giao [1,2] dài 1
  => True
rec1=[0, 0, 1, 1] rec2=[1, 0, 2, 1]
  trục x: đoạn [0,1] và [1,2] -> giao [1,1] dài 0
  trục y: đoạn [0,1] và [0,1] -> giao [0,1] dài 1
  => False
rec1=[0, 0, 1, 1] rec2=[2, 2, 3, 3]
  trục x: đoạn [0,1] và [2,3] -> giao [2,1] dài -1
  trục y: đoạn [0,1] và [2,3] -> giao [2,1] dài -1
  => False
rec1=[0, 0, 10, 10] rec2=[2, 3, 4, 5]
  trục x: đoạn [0,10] và [2,4] -> giao [2,4] dài 2
  trục y: đoạn [0,10] và [3,5] -> giao [3,5] dài 2
  => True


### 7.2. Trace 4 điều kiện tách rời cho ca "chạm góc"

In [11]:
def trace_separation(rec1, rec2):
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    conds = {
        "trái  (ax2 <= bx1)": ax2 <= bx1,
        "phải  (ax1 >= bx2)": ax1 >= bx2,
        "dưới  (ay2 <= by1)": ay2 <= by1,
        "trên  (ay1 >= by2)": ay1 >= by2,
    }
    for name, val in conds.items():
        print(f"  {name}: {val}")
    res = not any(conds.values())
    print("  => chồng nhau?", res)
    return res

trace_separation([0, 0, 1, 1], [1, 1, 2, 2]);   # chạm đúng một góc (1,1)

  trái  (ax2 <= bx1): True
  phải  (ax1 >= bx2): False
  dưới  (ay2 <= by1): True
  trên  (ay1 >= by2): False
  => chồng nhau? False


`any(iterable)` trả `True` nếu **có ít nhất một** phần tử đúng — tương đương nối các điều kiện bằng `or`. `dict.items()` cho các cặp `(key, value)`.

## 8. Code nộp LeetCode

In [12]:
class Solution:
    def isRectangleOverlap(self, rec1: list[int], rec2: list[int]) -> bool:
        ax1, ay1, ax2, ay2 = rec1
        bx1, by1, bx2, by2 = rec2
        return min(ax2, bx2) > max(ax1, bx1) and min(ay2, by2) > max(ay1, by1)

sol = Solution()
assert sol.isRectangleOverlap([0, 0, 2, 2], [1, 1, 3, 3]) is True
assert sol.isRectangleOverlap([0, 0, 1, 1], [1, 0, 2, 1]) is False
assert sol.isRectangleOverlap([0, 0, 1, 1], [2, 2, 3, 3]) is False
print("Solution PASS 3 ví dụ đề")

Solution PASS 3 ví dụ đề


Bản theo điều kiện tách rời (cũng nộp được):

In [13]:
class SolutionSeparation:
    def isRectangleOverlap(self, rec1: list[int], rec2: list[int]) -> bool:
        return not (rec1[2] <= rec2[0] or   # rec1 bên trái
                    rec1[0] >= rec2[2] or   # rec1 bên phải
                    rec1[3] <= rec2[1] or   # rec1 bên dưới
                    rec1[1] >= rec2[3])     # rec1 bên trên

print(SolutionSeparation().isRectangleOverlap([0, 0, 2, 2], [1, 1, 3, 3]))

True


## 9. Vì sao đúng? Độ phức tạp

**Bổ đề:** Một điểm `(x, y)` thuộc **phần trong** của cả hai hình ⇔ `x` thuộc phần trong của cả hai đoạn chiếu x **và** `y` thuộc phần trong của cả hai đoạn chiếu y. Vì hình chữ nhật song song trục chính là **tích** của hai đoạn: `[x1, x2] × [y1, y2]`.

Do đó phần giao của hai hình là `[max(x1), min(x2)] × [max(y1), min(y2)]` (nếu các đoạn hợp lệ), với diện tích = `(giao x) × (giao y)`. Diện tích dương ⇔ cả hai thừa số dương ⇔ điều kiện của cách 3. Cách 2 tương đương theo De Morgan.

**Độ phức tạp:** Time `O(1)`, Space `O(1)` — chỉ vài phép so sánh, không phụ thuộc độ lớn tọa độ.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đếm ô | `O(W·H)` | `O(1)` | Chỉ minh họa / oracle cho tọa độ nhỏ |
| 2. Phủ định 4 phía | `O(1)` | `O(1)` | Trực quan "nằm hẳn một bên" |
| 3. Chiếu hai trục | `O(1)` | `O(1)` | **Gọn, tổng quát** |
| 4. Diện tích giao | `O(1)` | `O(1)` | Cẩn thận hai độ dài cùng âm |

## 10. Cách 4 — Tính diện tích giao (và cái bẫy)

Ý tưởng: diện tích giao = `max(0, giao x) * max(0, giao y)`; chồng nhau ⇔ diện tích `> 0`.

Phải có `max(0, ...)`: nếu bỏ đi, hai độ dài **cùng âm** nhân nhau ra **dương** → sai (ví dụ 3 của đề!).

In [14]:
def overlap_area(rec1, rec2):
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    w = max(0, min(ax2, bx2) - max(ax1, bx1))
    h = max(0, min(ay2, by2) - max(ay1, by1))
    return w * h > 0

def overlap_area_wrong(rec1, rec2):
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    w = min(ax2, bx2) - max(ax1, bx1)
    h = min(ay2, by2) - max(ay1, by1)
    return w * h > 0                       # (-1) * (-1) = 1 > 0 !

print("đúng:", overlap_area([0, 0, 1, 1], [2, 2, 3, 3]))
print("sai :", overlap_area_wrong([0, 0, 1, 1], [2, 2, 3, 3]))

đúng: False
sai : True


Trong Python số nguyên không tràn, nhưng ở C++/Java tích `w * h` với tọa độ `10^9` có thể tràn `int` — thêm một lý do ưu tiên so sánh (cách 3) thay vì nhân.

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ đề.
2. Chạm cạnh dọc, cạnh ngang, chạm góc → `False`.
3. Một hình **chứa** hình kia → `True`.
4. Hai hình **trùng** nhau → `True`.
5. Hình "chữ thập" (giao nhau mà không góc nào nằm trong hình kia) → `True`.
6. Tọa độ âm và rất lớn.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.isRectangleOverlap([0, 0, 2, 2], [1, 1, 3, 3]), True)
check("đề 2 chạm cạnh dọc", sol.isRectangleOverlap([0, 0, 1, 1], [1, 0, 2, 1]), False)
check("đề 3 tách rời", sol.isRectangleOverlap([0, 0, 1, 1], [2, 2, 3, 3]), False)
check("chạm cạnh ngang", sol.isRectangleOverlap([0, 0, 2, 1], [0, 1, 2, 2]), False)
check("chạm góc", sol.isRectangleOverlap([0, 0, 1, 1], [1, 1, 2, 2]), False)
check("chứa nhau", sol.isRectangleOverlap([0, 0, 10, 10], [2, 3, 4, 5]), True)
check("trùng nhau", sol.isRectangleOverlap([1, 1, 4, 4], [1, 1, 4, 4]), True)
check("chữ thập", sol.isRectangleOverlap([0, 2, 6, 4], [2, 0, 4, 6]), True)
check("tọa độ âm", sol.isRectangleOverlap([-5, -5, -1, -1], [-3, -3, 0, 0]), True)
check("tọa độ lớn", sol.isRectangleOverlap([-10**9, -10**9, 10**9, 10**9], [10**9 - 1, 0, 10**9, 1]), True)

PASS đề 1: True
PASS đề 2 chạm cạnh dọc: False
PASS đề 3 tách rời: False
PASS chạm cạnh ngang: False
PASS chạm góc: False
PASS chứa nhau: True
PASS trùng nhau: True
PASS chữ thập: True
PASS tọa độ âm: True
PASS tọa độ lớn: True


Ca "chữ thập" là **phản ví dụ** cho một ý tưởng sai phổ biến: "chồng nhau ⇔ có một góc của hình này nằm trong hình kia". Ở đây không góc nào nằm trong hình kia nhưng vẫn chồng nhau.

In [16]:
def corner_inside_idea(rec1, rec2):            # Ý TƯỞNG SAI
    def inside(px, py, r):
        return r[0] < px < r[2] and r[1] < py < r[3]
    def corners(r):
        return [(r[0], r[1]), (r[0], r[3]), (r[2], r[1]), (r[2], r[3])]
    return any(inside(x, y, rec2) for x, y in corners(rec1)) or \
           any(inside(x, y, rec1) for x, y in corners(rec2))

print("ý tưởng 'góc nằm trong':", corner_inside_idea([0, 2, 6, 4], [2, 0, 4, 6]), "| đúng là:", True)

ý tưởng 'góc nằm trong': False | đúng là: True


### 11.1. Đối chiếu với oracle đếm ô trên tọa độ nhỏ ngẫu nhiên (có seed)

In [17]:
import random
random.seed(836)

def rand_rect():
    x1, x2 = sorted(random.sample(range(-4, 5), 2))   # sample: 2 số khác nhau -> x1 < x2
    y1, y2 = sorted(random.sample(range(-4, 5), 2))
    return [x1, y1, x2, y2]

impls = [overlap_separation, overlap_projection, overlap_area,
         Solution().isRectangleOverlap, SolutionSeparation().isRectangleOverlap]
count_true = 0
for _ in range(3000):
    r1, r2 = rand_rect(), rand_rect()
    expected = overlap_grid(r1, r2)
    count_true += expected
    for f in impls:
        assert f(r1, r2) == expected, (r1, r2, f)
    assert Solution().isRectangleOverlap(r2, r1) == expected     # tính đối xứng
print(f"3000 cặp ngẫu nhiên khớp oracle ({count_true} cặp chồng nhau); kết quả đối xứng")

3000 cặp ngẫu nhiên khớp oracle (1409 cặp chồng nhau); kết quả đối xứng


## 12. Lỗi người mới hay gặp

### Lỗi 1: dùng `<` thay vì `<=` ở điều kiện tách rời (hoặc `>=` thay vì `>` ở cách 3)

Khi đó chạm cạnh bị coi là chồng nhau.

In [18]:
def wrong_touch(rec1, rec2):
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    return min(ax2, bx2) >= max(ax1, bx1) and min(ay2, by2) >= max(ay1, by1)   # SAI: >=

print("sai :", wrong_touch([0, 0, 1, 1], [1, 0, 2, 1]))
print("đúng:", Solution().isRectangleOverlap([0, 0, 1, 1], [1, 0, 2, 1]))

sai : True
đúng: False


### Lỗi 2: chỉ kiểm tra rec1 so với rec2 một chiều

Chỉ kiểm tra "rec1 bên trái" và "rec1 bên dưới" mà quên "bên phải", "bên trên".

In [19]:
def wrong_two_sides(rec1, rec2):
    return not (rec1[2] <= rec2[0] or rec1[3] <= rec2[1])

print("sai :", wrong_two_sides([5, 5, 6, 6], [0, 0, 1, 1]))
print("đúng:", Solution().isRectangleOverlap([5, 5, 6, 6], [0, 0, 1, 1]))

sai : True
đúng: False


### Lỗi 3: nhầm thứ tự tọa độ `[x1, y1, x2, y2]` thành `[x1, x2, y1, y2]`

In [20]:
def wrong_order(rec1, rec2):
    ax1, ax2, ay1, ay2 = rec1          # SAI thứ tự
    bx1, bx2, by1, by2 = rec2
    return min(ax2, bx2) > max(ax1, bx1) and min(ay2, by2) > max(ay1, by1)

print("sai :", wrong_order([0, 0, 2, 2], [1, 1, 3, 3]))
print("đúng:", Solution().isRectangleOverlap([0, 0, 2, 2], [1, 1, 3, 3]))

sai : False
đúng: True


### Lỗi 4: nhân hai độ dài giao mà không chặn `max(0, ·)` — đã thấy ở mục 10.

## 13. Mẹo nhận dạng pattern

- Hình song song trục (axis-aligned) → **tách theo từng trục**; bài 2D trở thành hai bài 1D về đoạn.
- Giao hai đoạn `[a1, a2]`, `[b1, b2]`: `[max(a1, b1), min(a2, b2)]`; rỗng khi đầu trái > đầu phải.
- Câu hỏi "có chồng không" thường dễ hơn khi nghĩ **phủ định** ("khi nào chắc chắn không chồng?") rồi dùng De Morgan.
- Đọc kỹ đề về **biên**: "chạm có tính không?" quyết định `<` hay `<=`.
- Bài liên quan: *Rectangle Area (223)*, *Merge Intervals (56)*, *Interval List Intersections (986)*, *Rectangle Area II (850)*.

## 14. Biến thể

### 14.1. Tính diện tích giao (dùng trong *Rectangle Area 223*)

In [21]:
def intersection_area(rec1, rec2) -> int:
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    w = max(0, min(ax2, bx2) - max(ax1, bx1))
    h = max(0, min(ay2, by2) - max(ay1, by1))
    return w * h

def union_area(rec1, rec2) -> int:
    area = lambda r: (r[2] - r[0]) * (r[3] - r[1])
    return area(rec1) + area(rec2) - intersection_area(rec1, rec2)

print(intersection_area([0, 0, 2, 2], [1, 1, 3, 3]), union_area([0, 0, 2, 2], [1, 1, 3, 3]))

1 7


`lambda r: ...` tạo một hàm nhỏ không tên — ở đây tương đương `def area(r): return ...`.

### 14.2. Chạm cũng tính là "giao" (đoạn đóng)

Chỉ cần đổi `>` thành `>=`.

In [22]:
def touch_or_overlap(rec1, rec2) -> bool:
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    return min(ax2, bx2) >= max(ax1, bx1) and min(ay2, by2) >= max(ay1, by1)

print(touch_or_overlap([0, 0, 1, 1], [1, 1, 2, 2]), touch_or_overlap([0, 0, 1, 1], [2, 2, 3, 3]))

True False


### 14.3. Bài 1D: hai đoạn thời gian (cuộc họp) có trùng nhau không?

Cuộc họp `[s1, e1)` và `[s2, e2)` trùng ⇔ `max(s1, s2) < min(e1, e2)` — chính là "nửa" của bài 836.

In [23]:
def meetings_conflict(a, b) -> bool:
    return max(a[0], b[0]) < min(a[1], b[1])

print(meetings_conflict([9, 10], [10, 11]), meetings_conflict([9, 11], [10, 12]))

False True


## 15. Bài tập (tự làm trước, đáp án ở cuối)

### Bài 1 — Tính tay
`rec1 = [-3, 0, 3, 4]`, `rec2 = [0, -1, 9, 2]`. Tính giao trục x, giao trục y, diện tích giao, và kết luận.

### Bài 2 — Dự đoán output
Không chạy, đoán: `max(0, -3)`, `not (False or True)`, `any([False, False])`, `1 <= 1 < 2`.

### Bài 3 — Điền TODO

In [24]:
def ex3_overlap(rec1: list[int], rec2: list[int]) -> bool:
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    # TODO: tính độ dài giao trên trục x và trục y, trả về True khi cả hai dương
    pass

# assert ex3_overlap([0, 0, 2, 2], [1, 1, 3, 3]) is True
# assert ex3_overlap([0, 0, 1, 1], [1, 0, 2, 1]) is False

### Bài 4 — Rectangle Area (LeetCode 223)
Tính tổng diện tích **phủ** bởi hai hình chữ nhật (phần chung chỉ tính một lần). Tham số: `ax1, ay1, ax2, ay2, bx1, by1, bx2, by2`.

In [25]:
def ex4_compute_area(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2) -> int:
    # TODO
    pass

# assert ex4_compute_area(-3, 0, 3, 4, 0, -1, 9, 2) == 45
# assert ex4_compute_area(-2, -2, 2, 2, -2, -2, 2, 2) == 16

### Bài 5 — Biến thể: hình chữ nhật giao (hoặc `None`)
Trả về `[x1, y1, x2, y2]` của phần giao nếu diện tích dương, ngược lại `None`.

In [26]:
def ex5_intersection_rect(rec1, rec2):
    # TODO
    pass

# assert ex5_intersection_rect([0, 0, 2, 2], [1, 1, 3, 3]) == [1, 1, 2, 2]
# assert ex5_intersection_rect([0, 0, 1, 1], [1, 0, 2, 1]) is None

### Bài 6 — Mini challenge: có cặp nào chồng nhau trong danh sách?
Cho list các hình chữ nhật (tối đa ~100 hình). Trả về `True` nếu **có ít nhất một cặp** chồng nhau.

In [27]:
def ex6_any_overlap(rects: list[list[int]]) -> bool:
    # TODO
    pass

# assert ex6_any_overlap([[0, 0, 1, 1], [1, 1, 2, 2], [3, 3, 5, 5]]) is False
# assert ex6_any_overlap([[0, 0, 1, 1], [4, 4, 6, 6], [5, 5, 7, 7]]) is True

## 16. Gợi ý 3 mức

**Mức 1 — hướng tư duy**
- Bài 1: dùng bảng ở mục 7.1.
- Bài 4: diện tích hợp = A + B − giao.
- Bài 6: thử mọi cặp `i < j`.

**Mức 2 — cú pháp**
- Bài 3: `min(ax2, bx2) - max(ax1, bx1) > 0`.
- Bài 5: tính 4 giá trị `max(ax1, bx1)`, `max(ay1, by1)`, `min(ax2, bx2)`, `min(ay2, by2)`.

**Mức 3 — gần code**
- Bài 4: `w = max(0, min(ax2, bx2) - max(ax1, bx1))`, tương tự `h`; `return (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - w*h`.
- Bài 6: `for i in range(n): for j in range(i + 1, n): if overlap(rects[i], rects[j]): return True`.

## 17. Đáp án tham khảo

**Bài 1:** giao x = `min(3, 9) - max(-3, 0)` = `3 - 0 = 3`; giao y = `min(4, 2) - max(0, -1)` = `2 - 0 = 2`; diện tích giao `6` > 0 ⇒ `True`.

**Bài 2:** `0`, `False`, `False`, `True`.

In [28]:
print(max(0, -3), not (False or True), any([False, False]), 1 <= 1 < 2)
assert Solution().isRectangleOverlap([-3, 0, 3, 4], [0, -1, 9, 2]) is True
assert intersection_area([-3, 0, 3, 4], [0, -1, 9, 2]) == 6

def ans3_overlap(rec1, rec2):
    ax1, ay1, ax2, ay2 = rec1
    bx1, by1, bx2, by2 = rec2
    return min(ax2, bx2) - max(ax1, bx1) > 0 and min(ay2, by2) - max(ay1, by1) > 0

def ans4_compute_area(ax1, ay1, ax2, ay2, bx1, by1, bx2, by2):
    w = max(0, min(ax2, bx2) - max(ax1, bx1))
    h = max(0, min(ay2, by2) - max(ay1, by1))
    return (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - w * h

def ans5_intersection_rect(rec1, rec2):
    x1, y1 = max(rec1[0], rec2[0]), max(rec1[1], rec2[1])
    x2, y2 = min(rec1[2], rec2[2]), min(rec1[3], rec2[3])
    if x2 > x1 and y2 > y1:
        return [x1, y1, x2, y2]
    return None

def ans6_any_overlap(rects):
    n = len(rects)
    for i in range(n):
        for j in range(i + 1, n):
            if ans3_overlap(rects[i], rects[j]):
                return True
    return False

assert ans3_overlap([0, 0, 2, 2], [1, 1, 3, 3]) is True
assert ans3_overlap([0, 0, 1, 1], [1, 0, 2, 1]) is False
assert ans4_compute_area(-3, 0, 3, 4, 0, -1, 9, 2) == 45
assert ans4_compute_area(-2, -2, 2, 2, -2, -2, 2, 2) == 16
assert ans5_intersection_rect([0, 0, 2, 2], [1, 1, 3, 3]) == [1, 1, 2, 2]
assert ans5_intersection_rect([0, 0, 1, 1], [1, 0, 2, 1]) is None
assert ans6_any_overlap([[0, 0, 1, 1], [1, 1, 2, 2], [3, 3, 5, 5]]) is False
assert ans6_any_overlap([[0, 0, 1, 1], [4, 4, 6, 6], [5, 5, 7, 7]]) is True
print("Đáp án bài 1-6 PASS")

0 False False True
Đáp án bài 1-6 PASS


## 18. Tổng kết — template nhớ nhanh và checklist submit

```python
ax1, ay1, ax2, ay2 = rec1
bx1, by1, bx2, by2 = rec2
return min(ax2, bx2) > max(ax1, bx1) and min(ay2, by2) > max(ay1, by1)
#      ^ giao trên trục x dương        ^ giao trên trục y dương
```

**Câu thần chú:** *"Chiếu xuống từng trục; giao đoạn = [max trái, min phải]; chạm không tính ⇒ dùng `>`."*

**Checklist trước khi submit:**
- [ ] Đúng thứ tự `[x1, y1, x2, y2]`?
- [ ] Dùng `>` (nghiêm ngặt) để chạm cạnh/góc trả `False`?
- [ ] Kiểm tra **cả hai** trục?
- [ ] Nếu nhân diện tích, đã chặn `max(0, ·)` cho từng chiều?
- [ ] Đã test chạm cạnh, chạm góc, chứa nhau, chữ thập?